# Isoglot 01: baseline answers

Generates Tiny Aya Global's answers to the 1,200 public TukaBench prompts
(harm, culture and benign sets in English, Hausa, Yoruba and Igbo) through AuditKIT.
No judging happens here; that is notebook 02.

Answers are made in 12 chunks (one per set and language) and each chunk is uploaded
to a private Hugging Face dataset as soon as it is done. If the run stops, run it
again: finished chunks are downloaded and skipped.

**Before running:**
1. Settings → Accelerator → **GPU T4 x2** (one GPU is used). Settings → Internet **on**.
2. Accept the terms at https://huggingface.co/CohereLabs/tiny-aya-global with your HF account.
3. Add-ons → Secrets → **`HF_TOKEN`** = a **Write** token (it creates the private
   results dataset). Never paste it into a cell.
4. Run it in the background: **Save Version → Save & Run All (Commit)**. It keeps
   running if you close the tab, and Kaggle keeps the log and outputs.


In [ ]:
%%capture
!pip install -q "isoglot[gpu] @ git+https://github.com/nftkingiii/isoglot"


In [ ]:
import os, time, torch
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import HfApi, auth_check, whoami
MODEL = "CohereLabs/tiny-aya-global"
auth_check(MODEL)  # raises if the gated terms are not accepted for this token

# Fail now, not after an hour of GPU time, if the token cannot write.
api = HfApi()
REPO = f"{whoami()['name']}/isoglot-runs"
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
print("access ok | results ->", REPO, "|", torch.cuda.get_device_name(0))


## Smoke check

8 prompts (2 per language from the harm set), once unbatched and once batched.
fp16 on a T4 can produce empty or garbled output for some models, and batching
can change answers if padding is wrong. The full run uses batching only if every
batched answer matches the unbatched one.


In [ ]:
from isoglot.data import load_tukabench
from isoglot.generate import generate, hf_model

smoke = load_tukabench(["afri-jbb-harm"], limit=2)
t0 = time.time(); one = generate(smoke, MODEL, max_tokens=128, batch_size=1); t1 = time.time()
many = generate(smoke, MODEL, max_tokens=128, batch_size=8); t2 = time.time()

same = [a.actual_output.strip() == b.actual_output.strip() for a, b in zip(one, many)]
for a, ok in zip(one, same):
    print(a.id, "| same when batched:", ok)
    print("  ", repr(a.actual_output[:160]))
print(f"unbatched {t1-t0:.0f}s, batched {t2-t1:.0f}s")
assert all(s.actual_output.strip() for s in one), "empty answers: try dtype=torch.float32"

BATCH = 8 if all(same) else 1
print("full run batch size:", BATCH)
del one, many; torch.cuda.empty_cache()


## Full run (resumable)

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import EntryNotFoundError
from isoglot import LANGS, SETS
from isoglot.generate import run_chunks

OUT = Path("/kaggle/working/runs")
MAX_TOKENS = 512

def remote(path):
    return path.relative_to(OUT).as_posix()

def fetch(path):
    try:
        hf_hub_download(REPO, remote(path), repo_type="dataset", local_dir=OUT)
        print("restored", remote(path), "from the Hub")
    except EntryNotFoundError:
        pass

def publish(path):
    api.upload_file(path_or_fileobj=str(path), path_in_repo=remote(path),
                    repo_id=REPO, repo_type="dataset")

model = hf_model(MODEL, batch_size=BATCH)
t0 = time.time()
merged = run_chunks(model, MODEL, OUT, sets=SETS, langs=LANGS, variant="base",
                    max_tokens=MAX_TOKENS, fetch=fetch, publish=publish)
publish(merged)
print(f"all chunks done in {(time.time()-t0)/60:.1f} min ->", REPO, remote(merged))


In [ ]:
import json
rows = [json.loads(l) for l in open(merged, encoding="utf-8")]
empty = sum(not (r["response"] or "").strip() for r in rows)
print(len(rows), "answers |", empty, "empty")
for lang in LANGS:
    r = next(r for r in rows if r["lang"] == lang and r["set"] == "afri-jbb-benign")
    print(lang, "|", r["prompt"][:70], "->", repr(r["response"][:120]))
